## Placement Readiness Intelligence System (PRIS)

## 1. Generating the Synthetic Training Dataset

In [1]:

import pandas as pd
import random as rd
import numpy as np

rd.seed(42)

ranges = {
    "Highly Ready": {"bucket_A": (80,100), "bucket_B": (0,3)},
    "Moderately Ready": {"bucket_A": (70,79), "bucket_B": (4,5)},
    "Needs Improvement": {"bucket_A": (51,69), "bucket_B": (6,8)},
    "Not Ready Yet": {"bucket_A": (0,50), "bucket_B": (9,10)}
}

bucket_a_cols = ['skill_match_percentage', 'critical_skill_match_percentage','project_relevance_score', 
                 'certification_relevance_score', 'internship_relevance_score', 'resume_completeness_score', 
                 'keyword_match_score', 'role_category_match_score']

bucket_b_cols = ['missing_skills_count','critical_missing_skills_count']

all_rows = []

for k in range(2000):
    row = {}
    x = rd.choice(list(ranges.keys()))

    bucket_a_range = ranges[x]['bucket_A']
    low = bucket_a_range[0]  
    high = bucket_a_range[1]

    bucket_b_range = ranges[x]['bucket_B']
    low_b = bucket_b_range[0]
    high_b = bucket_b_range[1]

    # Generate a random float between 0.0 and 1.0
    chance = rd.random()

    if chance < 0.9:
        # 90% chance: Keep the original label 'x'
        label_to_use = x
    else:
        # 10% chance: Choose a label OTHER than 'x'
        other_labels = [label for label in ranges.keys() if label != x]
        label_to_use = rd.choice(other_labels)

    for i in bucket_a_cols:
        row[i] = rd.uniform(low, high)

    for i in bucket_b_cols:
        row[i] = rd.randint(low_b, high_b)

    row["readiness_label"] = label_to_use
    all_rows.append(row)

df = pd.DataFrame(all_rows)

df.to_csv('student_data.csv', index=False)


## 2. Verifying the Generated Dataset

In [2]:
df = pd.read_csv('student_data.csv')
print(df.shape)
print(df['readiness_label'].value_counts())
df.head()

(2000, 11)
readiness_label
Highly Ready         512
Not Ready Yet        501
Needs Improvement    495
Moderately Ready     492
Name: count, dtype: int64


,skill_match_percentage,critical_skill_match_percentage,project_relevance_score,certification_relevance_score,internship_relevance_score,resume_completeness_score,keyword_match_score,role_category_match_score,missing_skills_count,critical_missing_skills_count,readiness_label
0,85.500586,84.464215,94.729424,93.533990,97.843591,81.738777,88.438436,80.595944,1,1,Highly Ready
1,94.320392,94.026499,88.390396,88.984181,85.563814,97.386006,95.176147,83.193186,3,2,Highly Ready
2,68.229835,57.058702,52.669425,52.740895,66.254899,61.867069,65.528309,64.135172,8,6,Needs Improvement
3,14.658914,31.431990,44.272587,18.081751,9.614430,3.477757,33.063166,38.653417,9,9,Not Ready Yet
4,89.068206,96.682209,83.253082,87.105414,93.403503,94.036406,93.670953,81.428050,1,1,Highly Ready


## 3. Loading Data and Creating Train/Test Split

In [3]:
df = pd.read_csv(r'D:\G_PRIS\student_data.csv')

X = df[['skill_match_percentage', 'critical_skill_match_percentage',
     'project_relevance_score', 'certification_relevance_score', 
     'internship_relevance_score', 'resume_completeness_score', 
     'keyword_match_score', 'role_category_match_score', 
     'missing_skills_count','critical_missing_skills_count']]

y = df['readiness_label']

from sklearn.model_selection import train_test_split 
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

## 4. Training the Logistic Regression Model

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
import warnings
import joblib

warnings.filterwarnings('ignore', category=UserWarning)

model = Pipeline([
    ('scaler', StandardScaler()),
    ('logistic', LogisticRegression(max_iter=3000))
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

joblib.dump(model, 'logistic_model.joblib')
print("Model saved successfully.")

                   precision    recall  f1-score   support

     Highly Ready       0.90      0.87      0.89        95
 Moderately Ready       0.76      0.93      0.84       100
Needs Improvement       0.91      0.78      0.84       111
    Not Ready Yet       0.88      0.84      0.86        94

         accuracy                           0.85       400
        macro avg       0.86      0.86      0.86       400
     weighted avg       0.86      0.85      0.86       400

Model saved successfully.


## 5. Training the Decision Tree Model

In [5]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score

# Create the Decision Tree model
decision_tree_model = DecisionTreeClassifier(
    random_state=42
)

# Train the model
decision_tree_model.fit(X_train, y_train)

# Make predictions on test data
y_pred_dt = decision_tree_model.predict(X_test)

# Evaluate the model
print("Decision Tree Classification Report:")
print(classification_report(y_test, y_pred_dt))

# Accuracy
dt_accuracy = accuracy_score(y_test, y_pred_dt)
print("Decision Tree Accuracy:", dt_accuracy)

Decision Tree Classification Report:
                   precision    recall  f1-score   support

     Highly Ready       0.78      0.78      0.78        95
 Moderately Ready       0.76      0.79      0.77       100
Needs Improvement       0.82      0.77      0.79       111
    Not Ready Yet       0.73      0.76      0.74        94

         accuracy                           0.77       400
        macro avg       0.77      0.77      0.77       400
     weighted avg       0.77      0.77      0.77       400

Decision Tree Accuracy: 0.7725


## 6. Training the Random Forest Model

Random Forest is an ensemble learning algorithm that combines multiple decision trees to improve prediction performance and reduce overfitting.

It is evaluated against the Logistic Regression baseline and Decision Tree model to determine whether an ensemble-based approach provides better classification performance for the PRIS readiness prediction task.

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# Create the Random Forest model
random_forest_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# Train the model
random_forest_model.fit(X_train, y_train)

# Make predictions
y_pred_rf = random_forest_model.predict(X_test)

# Evaluate the model
print("Random Forest Classification Report:")
print(classification_report(y_test, y_pred_rf))

rf_accuracy = accuracy_score(y_test, y_pred_rf)
print("Random Forest Accuracy:", rf_accuracy)

Random Forest Classification Report:
                   precision    recall  f1-score   support

     Highly Ready       0.91      0.93      0.92        95
 Moderately Ready       0.88      0.94      0.91       100
Needs Improvement       0.92      0.87      0.90       111
    Not Ready Yet       0.87      0.84      0.85        94

         accuracy                           0.90       400
        macro avg       0.89      0.90      0.89       400
     weighted avg       0.90      0.90      0.89       400

Random Forest Accuracy: 0.895


### Random Forest Performance

The Random Forest model achieved an accuracy of 89.5% and a macro F1-score of 0.89 on the test dataset. It performed better than both Logistic Regression and the Decision Tree model on the current synthetic dataset.

The relatively balanced F1-scores across the four readiness classes indicate that the model performs reasonably well across all classes rather than favoring only one class.

## 7. Training the Gradient Boosting Model

Gradient Boosting is an ensemble learning algorithm that builds decision trees sequentially. Each new tree focuses on correcting the errors made by the previous trees.

It is evaluated along with Logistic Regression, Decision Tree, and Random Forest to identify the best-performing model for the PRIS readiness classification task.

In [7]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, accuracy_score

gradient_boosting_model = GradientBoostingClassifier(
    random_state=42
)

gradient_boosting_model.fit(X_train, y_train)

y_pred_gb = gradient_boosting_model.predict(X_test)

print("Gradient Boosting Classification Report:")
print(classification_report(y_test, y_pred_gb))

gb_accuracy = accuracy_score(y_test, y_pred_gb)
print("Gradient Boosting Accuracy:", gb_accuracy)

Gradient Boosting Classification Report:
                   precision    recall  f1-score   support

     Highly Ready       0.89      0.93      0.91        95
 Moderately Ready       0.87      0.93      0.90       100
Needs Improvement       0.92      0.86      0.89       111
    Not Ready Yet       0.88      0.84      0.86        94

         accuracy                           0.89       400
        macro avg       0.89      0.89      0.89       400
     weighted avg       0.89      0.89      0.89       400

Gradient Boosting Accuracy: 0.89


## 8. Model Performance Comparison

The performance of the four classification models is compared below:

| Model | Accuracy | Macro Precision | Macro Recall | Macro F1-Score |
|---|---:|---:|---:|---:|
| Logistic Regression | 85.00% | 0.86 | 0.85 | 0.85 |
| Decision Tree | 77.25% | 0.77 | 0.77 | 0.77 |
| **Random Forest** | **89.50%** | **0.89** | **0.90** | **0.89** |
| Gradient Boosting | 89.00% | 0.89 | 0.89 | 0.89 |

### Model Selection

Random Forest achieved the highest accuracy of **89.50%** among the four evaluated models. It also achieved a macro F1-score of **0.89**, indicating balanced performance across the four readiness classes.

Therefore, **Random Forest is selected as the final classification model for the PRIS system**.

## 9. Saving the Trained Model

In [8]:
import joblib

joblib.dump(random_forest_model, 'random_forest_model.joblib')

print("Random Forest model saved successfully.")

Random Forest model saved successfully.


## 10. Installing Required Packages for the LLM Pipeline

In [9]:
%pip install python-dotenv
%pip install pypdf
%pip install groq

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 11. Setting Up the Groq API Client

In [10]:
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")
client = Groq(api_key=groq_api_key)

## 12. Extracting Text from a Sample Resume PDF

In [11]:
import pypdf

reader = pypdf.PdfReader("g_resume.pdf")
text = ""

for page in reader.pages:
    text += page.extract_text()

## 13. Sample Job Description for Testing

In [12]:
jd_text = '''This role delivers engaging online training sessions while providing 
            empathetic customer support to resolve daily technical and program inquiries. 
            You will guide learners through product onboarding, troubleshoot system issues 
            via chat and email, and create clear instructional materials to ensure student success. 
            The ideal candidate blends professional teaching or tutoring experience with a patient, 
            customer-first approach to problem-solving.'''

## 14. Defining Feature Order for Model Input

In [13]:
feature_order = ['skill_match_percentage', 'critical_skill_match_percentage',
     'project_relevance_score', 'certification_relevance_score', 
     'internship_relevance_score', 'resume_completeness_score', 
     'keyword_match_score', 'role_category_match_score', 
     'missing_skills_count','critical_missing_skills_count']

## 15. Final Analysis Function (Resume + JD → Prediction)

In [16]:
import json

def analyze_resume(resume_text, jd_text):

    prompt = f"""
                You are an expert placement evaluator.

                Analyze the candidate's resume against the given job description.

                Evaluate only the evidence present in the resume and the requirements mentioned in the job description.

                Scoring rules:
                - 90-100 = Excellent
                - 80-89 = Strong
                - 70-79 = Moderate
                - 50-69 = Weak
                - 0-49 = Very Weak

                Evaluate these features:

                1. skill_match_percentage:
                Percentage of relevant required skills from the job description that are demonstrated in the resume.

                2. critical_skill_match_percentage:
                Percentage of important/critical required skills from the job description that are demonstrated in the resume.

                3. project_relevance_score:
                How relevant the candidate's projects are to the job requirements.

                4. certification_relevance_score:
                How relevant the candidate's certifications are to the job requirements.

                5. internship_relevance_score:
                How relevant the candidate's internship/work experience is to the job requirements.

                6. resume_completeness_score:
                How complete and professionally structured the resume is.

                7. keyword_match_score:
                How well important job-related keywords appear in the resume.

                8. role_category_match_score:
                How well the candidate's overall background matches the role.

                Important consistency rules:

                - Only consider skills that are relevant to the given job description.
                - If a required skill is clearly present in the resume, treat it as a matched skill.
                - A skill cannot appear in both matched_skills and missing_skills.
                - A skill cannot appear in both matched_skills and critical_missing_skills.
                - Do not mark a skill as missing if it is clearly present in the resume.
                - Do not give a high skill_match_percentage when important required skills are missing.
                - Every skill in missing_skills must actually be missing from the resume.
                - Every skill in critical_missing_skills must actually be missing from the resume.
                - critical_missing_skills must be a subset of missing_skills.
                - missing_skills_count must equal the number of items in missing_skills.
                - critical_missing_skills_count must equal the number of items in critical_missing_skills.
                - Do not mention a skill as missing in feedback unless it appears in missing_skills.
                - Do not invent skills that are unrelated to the job description.
                - Do not describe optional or desirable skills as candidate gaps if they are not required by the job description.
                - Clearly distinguish required skill gaps from optional career-enhancement recommendations.
                - Focus feedback primarily on the explicit requirements of the job description.
                - Keep all numerical scores realistic and based on evidence from the resume and job description.

                Return ONLY a JSON object with these fields:

                {{
                    "skill_match_percentage": 75,
                    "critical_skill_match_percentage": 75,
                    "project_relevance_score": 75,
                    "certification_relevance_score": 75,
                    "internship_relevance_score": 75,
                    "resume_completeness_score": 75,
                    "keyword_match_score": 75,
                    "role_category_match_score": 75,
                    "missing_skills_count": 0,
                    "critical_missing_skills_count": 0,
                    "missing_skills": [],
                    "critical_missing_skills": [],
                    "matched_skills": [],
                    "feedback": "",
                    "roadmap_7_day": "",
                    "roadmap_30_day": "",
                    "Resume_improvement_suggestion": "",
                    "Job-specific_preparation_suggestions": ""
                }}

                Resume:
                {resume_text}

                Job Description:
                {jd_text}
                """
        
    # Step 2: call Groq
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content":prompt}],
        response_format={"type": "json_object"}
        )
        
    # Step 3: parse JSON
    data = json.loads(response.choices[0].message.content)
    print(data)

    # Step 4: fix count mismatches
    data["missing_skills"] = data.get("missing_skills", [])
    data["critical_missing_skills"] = data.get("critical_missing_skills", [])
    data["matched_skills"] = data.get("matched_skills", [])
    data["missing_skills_count"] = len(data["missing_skills"])
    data["critical_missing_skills_count"] = len(data["critical_missing_skills"])

    readiness_score = sum(data[col] for col in bucket_a_cols) / len(bucket_a_cols)
    data["placement_readiness_score"] = round(readiness_score, 2)
        
    # Step 5: build input row for the model
    input_row = pd.DataFrame([data])[feature_order]
        
    # Step 6: predict
    prediction = random_forest_model.predict(input_row.values)
                
        
    # Step 7: add prediction into data
    data["predicted_readiness_label"] = prediction[0]
        
    # Step 8: return everything
    return data

result = analyze_resume(text, jd_text)
print(json.dumps(result, indent=2))


{'skill_match_percentage': 88, 'critical_skill_match_percentage': 100, 'project_relevance_score': 70, 'certification_relevance_score': 50, 'internship_relevance_score': 85, 'resume_completeness_score': 70, 'keyword_match_score': 75, 'role_category_match_score': 80, 'missing_skills_count': 2, 'critical_missing_skills_count': 1, 'missing_skills': ['Experience with online training platforms', 'Experience creating digital instructional materials'], 'critical_missing_skills': ['Experience with online training platforms'], 'matched_skills': ['Teaching experience (grade 3-7)', 'Technical support experience (streaming device activation and troubleshooting)', 'Hardware/software and network troubleshooting', 'Strong communication and customer interaction', 'Empathetic student support', 'Problem‑solving under pressure', 'Lesson planning and creation of instructional content'], 'feedback': 'The candidate demonstrates strong teaching experience and solid technical support background, both of which 

In [15]:
from groq import Groq
import os

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

models = client.models.list()

for model in models.data:
    print(model.id)

openai/gpt-oss-20b
allam-2-7b
groq/compound-mini
meta-llama/llama-prompt-guard-2-86m
groq/compound
meta-llama/llama-prompt-guard-2-22m
whisper-large-v3-turbo
canopylabs/orpheus-v1-english
qwen/qwen3.6-27b
whisper-large-v3
qwen/qwen3.8-27b
openai/gpt-oss-safeguard-20b
openai/gpt-oss-120b
canopylabs/orpheus-arabic-saudi
